# Train YOLO11-seg on the cleaned CarDD dataset

**Before you run anything:** upload `dataset.zip` (147 MB, from the project folder) to
Google Drive at `MyDrive/vda/dataset.zip`. Create the `vda` folder first.

**Then:** Runtime → Change runtime type → **T4 GPU**. Without a GPU this will not finish.

Expect roughly **2h20m** for 100 epochs on a T4 (measured: 2.321 h on 31 Aug 2026, ~1m13s/epoch). Colab's free tier disconnects if
you leave the tab idle too long, so keep it open and check in occasionally. Every
checkpoint is mirrored to Drive, so a disconnect costs you one epoch, not the run.

**Why this is faster than it used to be.** The training split went from 6,918 images to
2,742 when the duplicate-photograph leak was fixed — 81.5% of the old test split was a
copy of a training photograph. An epoch is 2.5x shorter as a result, which is why the
epoch budget went up from 60 to 100 and the run still finishes sooner.

**You have not lost the augmentation.** Roboflow's copies were *offline* augmentation:
three fixed rotations of each photograph, baked into files, the same three every epoch.
Ultralytics augments *online* instead — mosaic, HSV shift, translate, scale and flip,
re-rolled randomly every epoch. 100 epochs of random augmentation over 2,742 photographs
is strictly more variety than 60 epochs over the same 2,742 photographs plus three frozen
rotations each. The offline copies were adding training time and destroying the ability to
measure anything, and buying very little.

**Why segmentation and not plain detection:** the CarDD labels are polygons, and a
polygon has an area. That area is what the severity module uses downstream — the dataset
has no severity labels, so severity has to be computed from how much of the panel is
affected. Training a box-only model would throw that away.

In [ ]:
# 1. Confirm there is actually a GPU attached.
import subprocess
print(subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total",
                      "--format=csv,noheader"], capture_output=True, text=True).stdout or
      "NO GPU FOUND -> Runtime > Change runtime type > T4 GPU, then rerun this cell")

In [ ]:
# 2. Install ultralytics (this is the package that provides YOLO11).
%pip install -q ultralytics
import ultralytics, torch
print("ultralytics", ultralytics.__version__, "| torch", torch.__version__,
      "| cuda", torch.cuda.is_available())

In [ ]:
# 3. Mount Drive (a link + permission prompt will appear).
from google.colab import drive
drive.mount("/content/drive")

from pathlib import Path
DRIVE = Path("/content/drive/MyDrive/vda")
DRIVE.mkdir(parents=True, exist_ok=True)
zip_path = DRIVE / "dataset.zip"
assert zip_path.exists(), f"Upload dataset.zip to {DRIVE} first"
print(f"found {zip_path} ({zip_path.stat().st_size/1024/1024:.0f} MB)")

In [ ]:
# 4. Unzip to Colab's LOCAL disk, not Drive.
#    Training reads every image once per epoch. Reading 3,933 small files off Drive over
#    the network would dominate the runtime; local disk makes it a non-issue.
import zipfile, shutil
from pathlib import Path

DATA = Path("/content/dataset")
if DATA.exists():
    shutil.rmtree(DATA)

# The zip is built with `cd dataset && zip -r ../dataset.zip .` (the command in README.md),
# so its members are images/, labels/ and data.yaml at the archive ROOT - there is no
# `dataset/` prefix. So extract INTO /content/dataset, not /content. Getting this wrong
# fails later, at `data.yaml not found`, with the files sitting safely in /content.
# The block below also tolerates an archive that does carry a `dataset/` prefix, so the
# notebook works whichever way the zip was built.
with zipfile.ZipFile("/content/drive/MyDrive/vda/dataset.zip") as z:
    z.extractall(DATA)

nested = DATA / "dataset"
if (nested / "data.yaml").exists() and not (DATA / "data.yaml").exists():
    for item in list(nested.iterdir()):
        shutil.move(str(item), str(DATA / item.name))
    nested.rmdir()

assert (DATA / "data.yaml").exists(), \
    f"data.yaml not found under {DATA}; got {[p.name for p in DATA.iterdir()]}"

# data.yaml ships with `path: .` (relative, so it works on any machine). Ultralytics
# resolves it against its own settings directory, not the yaml's location, so make it
# absolute here rather than relying on where Colab thinks the dataset root is.
yaml_path = DATA / "data.yaml"
text = yaml_path.read_text().replace("path: .", f"path: {DATA}")
yaml_path.write_text(text)
print(text)

for split in ("train", "valid", "test"):
    n_img = len(list((DATA / "images" / split).iterdir()))
    n_lbl = len(list((DATA / "labels" / split).iterdir()))
    print(f"  {split:<6} {n_img} images / {n_lbl} labels")
    assert n_img == n_lbl, "image/label mismatch - re-run tools/prepare_dataset.py"


In [ ]:
# 5. Train.
import shutil
from pathlib import Path
from ultralytics import YOLO

DRIVE = Path("/content/drive/MyDrive/vda")
RUN = Path("/content/runs/yolo11s-seg")

model = YOLO("yolo11s-seg.pt")   # COCO-pretrained weights; we fine-tune from these

def mirror_to_drive(trainer):
    """Copy the checkpoint to Drive after every save.

    Colab free sessions die without warning. Writing checkpoints straight to Drive
    would slow every epoch, so we train to local disk and mirror afterwards - the
    files are ~20 MB, so this costs a second or two per epoch and means a disconnect
    never loses more than one epoch of work.
    """
    # Wrapped in try/except deliberately: this runs inside the training loop, so an
    # exception here (a Drive hiccup, or an attribute rename in a future ultralytics)
    # would abort the whole run. Failing to mirror is recoverable; losing 3 hours is not.
    try:
        for name in ("last.pt", "best.pt"):
            src = Path(trainer.wdir) / name
            if src.exists():
                shutil.copy2(src, DRIVE / name)
        for name in ("results.csv", "args.yaml"):
            src = Path(trainer.save_dir) / name
            if src.exists():
                shutil.copy2(src, DRIVE / name)
        print("  [mirrored checkpoint to Drive]")
    except Exception as e:
        print(f"  [mirror to Drive failed, training continues] {e}")

model.add_callback("on_model_save", mirror_to_drive)

results = model.train(
    data="/content/dataset/data.yaml",
    epochs=100,         # ~2h20m on a T4 at 2,742 training images; patience stops it early
                        # if it stalls, so a generous budget costs nothing but time
    imgsz=640,          # the images are already 640 on their long edge, so no upscaling
    batch=16,           # fits comfortably in a T4's 15 GB at 640px for a seg model
    workers=2,          # Colab gives 2 vCPUs; more workers just thrash
    patience=15,        # stop early if val mAP stalls for 15 epochs
    seed=0,             # reproducible, so the numbers in the report can be regenerated
    project="/content/runs",
    name="yolo11s-seg",
    exist_ok=True,
    plots=True,         # writes the confusion matrix and PR curves you need as figures
)
print("\nbest weights:", RUN / "weights" / "best.pt")

## If the session disconnected mid-run

Reconnect, re-run cells 1–4, then run the cell below instead of cell 5. It pulls the
mirrored checkpoint back from Drive and picks up where the run stopped, keeping the
optimiser state and learning-rate schedule intact.

If `resume=True` complains, fall back to warm-starting a fresh run from the saved
weights: `YOLO("/content/drive/MyDrive/vda/last.pt").train(data=..., epochs=<remaining>)`.
That restarts the LR schedule, which is slightly worse but perfectly usable.

In [ ]:
# 6. Resume after a disconnect.
import shutil
from pathlib import Path
from ultralytics import YOLO

DRIVE = Path("/content/drive/MyDrive/vda")
RUN = Path("/content/runs/yolo11s-seg")
(RUN / "weights").mkdir(parents=True, exist_ok=True)
for name in ("last.pt", "best.pt"):
    if (DRIVE / name).exists():
        shutil.copy2(DRIVE / name, RUN / "weights" / name)
for name in ("results.csv", "args.yaml"):
    if (DRIVE / name).exists():
        shutil.copy2(DRIVE / name, RUN / name)

YOLO(str(RUN / "weights" / "last.pt")).train(resume=True)

In [ ]:
# 7. Evaluate on the HELD-OUT TEST split.
#    These are the numbers that go in the report. They mean something because
#    tools/prepare_dataset.py guaranteed no test photograph appears in training -
#    1,872 clusters straddled the original Roboflow boundary, and 81.5% of its test
#    split was a copy of a training photograph. That is now 0, asserted before the
#    dataset is written.
#
#    Read tire_flat's AP with care: only 32 test instances, so it swings between runs.
#    Report it as a range or with an explicit caveat, not as a point value.
#
#    This cell restores best.pt from Drive if /content/runs is empty, so it works after
#    a Colab runtime reset. That happens routinely: a fresh VM wipes /content AND the
#    pip installs, so `from ultralytics import YOLO` fails and the run folder is gone,
#    while Drive still has every mirrored checkpoint. Re-run cells 1-4 first to reinstall
#    ultralytics, re-mount Drive and re-unzip the dataset, then run this. Do NOT re-run
#    cell 5 - that starts a fresh 2h20m run and overwrites best.pt in Drive.
import shutil
from pathlib import Path
from ultralytics import YOLO

DRIVE = Path("/content/drive/MyDrive/vda")
RUN = Path("/content/runs/yolo11s-seg")
WEIGHTS = RUN / "weights" / "best.pt"

if not WEIGHTS.exists():
    src = DRIVE / "best.pt"
    assert src.exists(), (
        f"no local best.pt and none mirrored at {src} - nothing to evaluate. "
        "If Drive has no best.pt either, the weights are genuinely lost and "
        "training has to be re-run."
    )
    WEIGHTS.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy2(src, WEIGHTS)
    print(f"restored best.pt from Drive ({WEIGHTS.stat().st_size / 1e6:.1f} MB)")
else:
    print("best.pt found locally")

# results.csv holds the full per-epoch history - the source for the report's loss and
# mAP curves. Bring it back too, since it is easy to lose and impossible to recreate
# without retraining.
for name in ("results.csv", "args.yaml"):
    if (DRIVE / name).exists() and not (RUN / name).exists():
        RUN.mkdir(parents=True, exist_ok=True)
        shutil.copy2(DRIVE / name, RUN / name)
        print(f"restored {name} from Drive")

model = YOLO(str(WEIGHTS))
m = model.val(data="/content/dataset/data.yaml", split="test", imgsz=640, plots=True)

print(f"\n{'':<18}{'box mAP50':>10}{'box mAP50-95':>14}{'mask mAP50':>12}{'mask mAP50-95':>15}")
print(f"{'OVERALL':<18}{m.box.map50:>10.3f}{m.box.map:>14.3f}{m.seg.map50:>12.3f}{m.seg.map:>15.3f}")
try:
    names = m.names if isinstance(m.names, dict) else dict(enumerate(m.names))
    for i, name in names.items():
        print(f"{name:<18}{m.box.ap50[i]:>10.3f}{m.box.maps[i]:>14.3f}"
              f"{m.seg.ap50[i]:>12.3f}{m.seg.maps[i]:>15.3f}")
except Exception as e:
    print("per-class breakdown unavailable:", e)
    print("the per-class table is also printed by val() above, and saved in the run folder")


In [ ]:
# 8. Save everything worth keeping to Drive: weights, metrics, and the plots you
#    will paste into the report (confusion matrix, PR curve, sample predictions).
import shutil
from pathlib import Path

DRIVE = Path("/content/drive/MyDrive/vda")
for run in Path("/content/runs").glob("*"):
    dest = DRIVE / "runs" / run.name
    if dest.exists():
        shutil.rmtree(dest)
    shutil.copytree(run, dest)
    print("copied", run.name, "->", dest)

print("\nDownload best.pt from Drive and put it in the project at:")
print("  vehicle damage assessment/weights/best.pt")

In [ ]:
# 9. Optional: a few annotated test predictions, useful as report figures.
import random
from pathlib import Path
from ultralytics import YOLO
from IPython.display import Image as Show, display

model = YOLO("/content/runs/yolo11s-seg/weights/best.pt")
test_images = sorted(Path("/content/dataset/images/test").iterdir())
random.seed(0)
for p in random.sample(test_images, 6):
    r = model.predict(str(p), imgsz=640, conf=0.35, save=True,
                      project="/content/preds", name="samples", exist_ok=True, verbose=False)
    display(Show(filename=str(Path(r[0].save_dir) / p.name), width=420))